In [ ]:
# pandas: tabular data loading and row-level iteration over pass records
# matplotlib.pyplot: figure and axes rendering
# mplsoccer.Pitch: pitch renderer used to overlay the pass visualization on a football pitch
# seaborn: statistical visualization library; used here specifically for sns.kdeplot(),
#   which computes and renders a 2D kernel density estimate over pass origin coordinates

import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import seaborn as sns

# Load the custom CSV containing Messi's pass events in the second half vs Real Betis.
# This is not a StatsBomb-formatted file — coordinates are stored in a raw 100x100-ish system
# and must be rescaled to the StatsBomb 120x80 pitch coordinate system before plotting.
df = pd.read_csv('messibetis6.csv')

# Rescale coordinates from the source system to StatsBomb's 120x80 pitch dimensions:
#   x * 1.2 maps the raw x-range to [0, 120] (pitch length in StatsBomb units)
#   y * 0.8 maps the raw y-range to [0,  80] (pitch width in StatsBomb units)
# Both pass origin (x, y) and pass destination (endX, endY) must be scaled consistently.
df['x']    = df['x']    * 1.2
df['y']    = df['y']    * 0.8
df['endX'] = df['endX'] * 1.2
df['endY'] = df['endY'] * 0.8

df

In [ ]:
# Set up a dark-themed StatsBomb pitch.
# pitch_type='statsbomb' tells mplsoccer to use the 120x80 coordinate system,
# matching the rescaled coordinates computed in the previous cell.
# pitch_color and line_color are standard dark-background aesthetic choices
# for player-specific visualizations.
pitch = Pitch(pitch_type='statsbomb', pitch_color='#22312b', line_color='#c7d5cc')
fig, ax = pitch.draw(figsize=(16, 11), constrained_layout=True, tight_layout=False)
fig.set_facecolor('#22312b')

# Iterate over every pass row and draw two overlapping elements per pass:
#   1. plt.plot() draws a line from the pass origin (x, y) to the destination (endX, endY),
#      representing the pass trajectory.
#   2. plt.scatter() places a dot at the pass origin, marking where Messi received/played the ball.
# Yellow encodes successful passes; red encodes unsuccessful ones.
# This outcome-coded visual immediately reveals spatial patterns in Messi's pass success rate.
for x in range(len(df['x'])):
    if df['outcome'][x] == 'Successful':
        plt.plot((df['x'][x], df['endX'][x]), (df['y'][x], df['endY'][x]), color='yellow')
        plt.scatter(df['x'][x], df['y'][x], c='yellow')
    if df['outcome'][x] == 'Unsuccessful':
        plt.plot((df['x'][x], df['endX'][x]), (df['y'][x], df['endY'][x]), color='red')
        plt.scatter(df['x'][x], df['y'][x], c='red')

# plt.xlim/ylim constrain the display to the rescaled StatsBomb pitch boundaries.
# plt.gca().invert_yaxis() flips the y-axis so the pitch renders in the conventional
# top-to-bottom orientation (origin at top-left, goal at top-right for the attacking team).
plt.xlim(0, 120)
plt.ylim(0, 80)
plt.title('Messi vs Betis, Heatmap passes', color='white', size=20)
pitch.draw(ax=ax)
plt.gca().invert_yaxis()

# Overlay a 2D kernel density estimate (KDE) on top of the pass map.
# sns.kdeplot() fits a Gaussian KDE to the (x, y) pass origin coordinates.
# fill=True fills the density contours with color; thresh=False renders all contours
# down to zero density rather than cutting off at a threshold.
# n_levels=10 controls how many isodensity contours are drawn.
# alpha=0.5 blends the KDE layer semi-transparently over the pass arrows.
# cmap='magma' provides a dark-to-bright gradient appropriate for dark backgrounds.
# The resulting overlay reveals Messi's dominant zones of operation regardless
# of pass outcome, capturing where he consistently touched the ball.
kde = sns.kdeplot(
    x=df['x'],
    y=df['y'],
    fill=True,
    thresh=False,
    alpha=0.5,
    n_levels=10,
    cmap='magma'
)

## Summary: Individual Player Pass Map with KDE Heatmap

### What This Notebook Does

This notebook produces a combined pass map and kernel density heatmap for a single player — Messi in the second half against Real Betis. It overlays three layers of information on a dark-themed pitch: outcome-coded pass trajectories (lines), pass origins (scatter dots), and a 2D KDE density surface that reveals the player's dominant zones of operation across all touches.

### Key Concepts

- **Coordinate rescaling**: The source CSV uses a non-StatsBomb coordinate system. Multiplying `x` by 1.2 and `y` by 0.8 transforms coordinates from the raw system into the StatsBomb 120x80 pitch space. Both origin (`x`, `y`) and destination (`endX`, `endY`) must be scaled with the same factors to preserve geometric relationships (pass angle and length).
- **Outcome-coded pass lines**: Each pass is drawn as a `plt.plot()` line from origin to destination. Yellow encodes successful passes; red encodes unsuccessful ones. The spatial distribution of red lines identifies zones where Messi's passing broke down — typically under high pressure or in tight channels.
- **`plt.gca().invert_yaxis()`**: StatsBomb coordinates place `y=0` at the bottom of the pitch. Inverting the y-axis renders the pitch in the conventional portrait orientation where the attacking direction reads left-to-right or bottom-to-top depending on the layout.
- **`sns.kdeplot()` with `fill=True`**: A 2D Gaussian KDE is fitted to all pass origin coordinates regardless of outcome. `n_levels=10` draws 10 isodensity contours. `thresh=False` includes all density levels down to near-zero, ensuring the entire pitch surface is covered. `cmap='magma'` maps density from dark purple (low) to bright yellow (high), making peak activity zones immediately visible against the dark background.
- **Layer ordering**: The KDE is drawn last and sits on top of the pass arrows. The `alpha=0.5` transparency preserves the underlying pass lines while the density gradient adds a statistical summary.

### Data Available

The `df` DataFrame contains Messi's pass events with rescaled coordinates:

| Column | Description |
|---|---|
| `x`, `y` | Rescaled pass origin in StatsBomb 120x80 coordinates |
| `endX`, `endY` | Rescaled pass destination |
| `outcome` | `Successful` or `Unsuccessful` |
| `minute`, `second` | Temporal position of each pass |

### Ideas to Extract More Value

- **Pass direction analysis**: Compute the angle of each pass vector (`arctan2(endY - y, endX - x)`) and group by zone to identify whether Messi tends to pass forward, laterally, or back from each area of the pitch.
- **Successful vs unsuccessful KDE comparison**: Render two separate KDE layers — one for successful passes (e.g., `cmap='YlGn'`) and one for unsuccessful passes (e.g., `cmap='Reds'`) — to spatially isolate failure zones from dominant-touch zones.
- **Progressive pass highlighting**: Flag passes where `endX > x + 10` (advancing at least 10 units toward goal) and color these distinctly to identify Messi's forward-progression contribution from different areas.
- **Multi-match aggregation**: Load and concatenate multiple match CSVs (ensuring consistent coordinate scaling across sources) to build a season-level touch and pass map that removes single-match variance.
- **Temporal split**: Separate the DataFrame into time windows (e.g., first 15 minutes vs last 15 minutes) and compare KDE heatmaps to identify positional drift or fatigue-related drops in activity zone.